In [ ]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import json

import numpy as np

In [ ]:
# Font
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["CMU Serif Roman"] + plt.rcParams["font.serif"]
plt.rcParams["font.size"] = 16

In [ ]:
YEAR = 2019
YEAR = 2024

import os
WORKING_DIR = os.environ.get("REPO_ROOT", os.path.abspath(".."))  # repo root (notebooks run from notebooks/)
DATA_DIR = f"{WORKING_DIR}/data"
IMG_DIR = f"{WORKING_DIR}/images"
REGRESSION_DIR = f"{DATA_DIR}/dirichlet/{YEAR}"

In [ ]:
fd = open(
    f"{DATA_DIR}/parties_description/political_parties_description_europe_2019.json",
    "r",
)
political_parties_description_2019 = json.load(fd)
fd.close()

fd = open(
    f"{DATA_DIR}/parties_description/political_parties_description_europe_2024.json",
    "r",
)
political_parties_description_2024 = json.load(fd)
fd.close()

In [ ]:
df_r2_2019 = pd.read_pickle(f"{DATA_DIR}/df_r2_scores_2019.pkl")
df_r2_2024 = pd.read_pickle(f"{DATA_DIR}/df_r2_scores_2024.pkl")

In [ ]:
list(df_r2_2019.columns)

In [ ]:
list(df_r2_2024.columns)

In [ ]:
rename_parties_2019 = {
    "Coal. Renaissance": "Coal. Besoin d'Europe",
    "Coal. Envie d'Europe": "Coal. Réveiller l'Europe",
}

df_r2_2019 = df_r2_2019.rename(columns=rename_parties_2019)
common_parties = set(df_r2_2019.columns).intersection(set(df_r2_2024.columns))
common_parties = list(common_parties)

color_party = {}
for party in political_parties_description_2024:
    party_name = political_parties_description_2024[party]["name"]
    color_party[party_name] = political_parties_description_2024[party]["color"]

In [ ]:
relationships = {}

for party in common_parties:
    relationships[party] = {
        "color": color_party[party],
        2019: {
            "combined": float(df_r2_2019.iloc[0][party]),
            "socioeconomic": float(df_r2_2019.iloc[1][party]),
            "app": float(df_r2_2019.iloc[2][party]),
        },
        2024: {
            "combined": float(df_r2_2024.iloc[0][party]),
            "socioeconomic": float(df_r2_2024.iloc[1][party]),
            "app": float(df_r2_2024.iloc[2][party]),
        },
    }

In [ ]:
order = [
    "La France Insoumise",
    "Europe Écologie",
    "Coal. Besoin d'Europe",
    "Les Républicains",
    "Rassemblement National",
][::-1]

In [ ]:
# Year-over-year change in mean predictive correlation per model.
# Mean is taken over ALL parties present in each election (NOT restricted to common
# parties): a per-party matched comparison does not generalize when the party set
# changes across elections (e.g. 2024 has "La France fière", absent in 2019), whereas
# the mean per year is a robust year-level summary. Decision 2026-06-03 (keep all-parties).
change_across_years = {
    "Socioeconomic": {
        2019: np.mean(df_r2_2019.iloc[1]),
        2024: np.mean(df_r2_2024.iloc[1]),
        "relative": (np.mean(df_r2_2024.iloc[1]) - np.mean(df_r2_2019.iloc[1])) / np.mean(df_r2_2019.iloc[1]) * 100,
    },
    "Mobile Services": {
        2019: np.mean(df_r2_2019.iloc[2]),
        2024: np.mean(df_r2_2024.iloc[2]),
        "relative": (np.mean(df_r2_2024.iloc[2]) - np.mean(df_r2_2019.iloc[2])) / np.mean(df_r2_2019.iloc[2]) * 100,
    },
    "All": {
        2019: np.mean(df_r2_2019.iloc[0]),
        2024: np.mean(df_r2_2024.iloc[0]),
        "relative": (np.mean(df_r2_2024.iloc[0]) - np.mean(df_r2_2019.iloc[0])) / np.mean(df_r2_2019.iloc[0]) * 100,
    },
}

change_across_years

In [ ]:
change_across_years

In [ ]:
color_2019 = "tab:blue"
color_2024 = "steelblue"
text_fontsize = 28
ticks_fontsize = 32
width = 0.42
delta = 0.45


plt.figure(figsize=(16, 11))

ax = plt.gca()

ax.grid(axis="y", linestyle="-", alpha=0.2, zorder=-5)

ax.spines["left"].set_color(color_2019)
ax.yaxis.label.set_color(color_2019)
# ax.tick_params(axis='y', colors=color_2019)
plt.ylabel(
    rf"Correlation ($\rho$)",
    color=color_2019,
    fontsize=text_fontsize,
    fontweight="bold",
)

for model_index, model in enumerate(change_across_years):
    model_2019 = change_across_years[model][2019]
    model_2024 = change_across_years[model][2024]

    plt.bar(
        2 * model_index - delta,
        model_2019 - 0.45,
        bottom=0.45,
        color=color_2019,
        alpha=0.5,
        width=width,
    )
    plt.text(
        2 * model_index - delta,
        0.1 + 0.36,
        "2019",
        ha="center",
        va="bottom",
        rotation=90,
        fontsize=text_fontsize,
        color="white",
        fontweight="bold",
    )
    plt.text(
        2 * model_index - delta,
        model_2019 + 0.01,
        f"{model_2019:.2f}",
        ha="center",
        fontsize=text_fontsize,
        alpha=0.9,
    )

    plt.bar(
        2 * model_index,
        model_2024 - 0.45,
        bottom=0.45,
        color=color_2024,
        alpha=0.8,
        width=width,
    )
    plt.text(
        2 * model_index,
        0.1 + 0.36,
        "2024",
        ha="center",
        va="bottom",
        rotation=90,
        fontsize=text_fontsize,
        color="white",
        fontweight="bold",
    )
    plt.text(
        2 * model_index,
        model_2024 + 0.01,
        f"{model_2024:.2f}",
        ha="center",
        fontsize=text_fontsize,
        alpha=0.9,
    )




plt.yticks(
    [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1],
    ["0", "0.1", "0.2", None, None, "0.5", "0.6", "0.7", "0.8", "0.9", "1"],
    fontsize=ticks_fontsize,
)

ax.set_ylim(-0.2, 1)
ax.set_ylim(0.35, 1)

plt.xticks(
    ticks=[0, 2, 4],
    labels=["Socioeconomic", "Mobile Services", "All"],
    fontsize=ticks_fontsize,
)
plt.xlim(-0.75, 4.75)

ax = plt.twinx()

color = "tab:grey"
ax.yaxis.label.set_color(color)
# ax.tick_params(axis='y', width=2)
plt.ylabel(rf"Relative Gain", color=color, fontsize=ticks_fontsize, fontweight="bold")


for model_index, model in enumerate(change_across_years):
    relative = change_across_years[model]["relative"]

    plt.bar(2 * model_index + delta, relative, color=color, alpha=0.8, width=width)

    if relative >= 0:
        plt.text(
            2 * model_index + delta,
            relative + 0.2,
            f"{relative:.2f}%",
            ha="center",
            va="bottom",
            fontsize=text_fontsize,
            alpha=0.9,
        )
    else:
        plt.text(
            2 * model_index + delta,
            relative - 0.2,
            f"{relative:.2f}%",
            ha="center",
            va="top",
            fontsize=text_fontsize,
            alpha=0.9,
        )

plt.yticks(
    [-2, 0, 2, 4, 6, 8, 10, 12, 14],
    ["-2%", "0%", "2%", "4%", "6%", "8%", "10%", "12%", "14%"],
    fontsize=ticks_fontsize,
)
plt.ylim(-2.5, 14)

plt.axhline(y=0, color="black", linestyle="-", alpha=0.7, zorder=-1)

plt.savefig(f'{IMG_DIR}/dirichlet_regression/change_rho_2019_2024.pdf', dpi=300, bbox_inches='tight')
plt.show()